# 04 - Quality Control & Preprocessing

## Learning objectives
1. Compute per-spot QC: total counts, genes detected, mitochondrial fraction.
2. Visualize QC distributions and QC over the histology.
3. Filter low-quality spots and rarely expressed genes.
4. Normalize, log-transform, select highly variable genes, scale, and run PCA.
5. Save a processed AnnData and a QC summary CSV.

## Concept
Raw UMI counts are noisy and unnormalized: spots differ in how much mRNA was captured
(sequencing depth), just as scans differ in exposure/intensity scaling. QC + normalization
is the expression analogue of intensity normalization, denoising, and masking out bad
voxels before analysis.


In [ ]:
# --- Standard setup: make `utils` importable and seed RNGs ---
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'utils').exists():
    ROOT = ROOT.parent  # in case the notebook is opened from a subfolder
sys.path.insert(0, str(ROOT))

from utils import st_helpers as st
st.set_seeds()  # reproducibility (seed = 0)
print('Project root:', st.project_root())


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

adata = st.load_adata('adata_raw.h5ad')
adata.var_names_make_unique()
adata.shape


### Step 1 - QC metrics
We flag **mitochondrial genes** (mouse names start with `mt-`; human `MT-`). A high
mitochondrial fraction often marks stressed/dying or low-quality spots. `calculate_qc_metrics`
adds `total_counts` (UMIs per spot), `n_genes_by_counts` (genes detected per spot), and
`pct_counts_mt`.

In [ ]:
# Detect mito genes for either species naming convention.
adata.var['mt'] = adata.var_names.str.lower().str.startswith('mt-')
print('mitochondrial genes found:', int(adata.var['mt'].sum()))

sc.pp.calculate_qc_metrics(
    adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True
)
adata.obs[['total_counts', 'n_genes_by_counts', 'pct_counts_mt']].describe()


**Expected output:** summary stats; median `total_counts` typically in the thousands and
`n_genes_by_counts` in the low thousands for brain tissue.

### Step 2 - QC plots (histograms + spatial)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(adata.obs['total_counts'], bins=60)
axes[0].set(title='Total UMI counts per spot', xlabel='total_counts', ylabel='# spots')
axes[1].hist(adata.obs['n_genes_by_counts'], bins=60, color='tab:green')
axes[1].set(title='Genes detected per spot', xlabel='n_genes_by_counts')
axes[2].hist(adata.obs['pct_counts_mt'], bins=60, color='tab:red')
axes[2].set(title='Mitochondrial % per spot', xlabel='pct_counts_mt')
plt.tight_layout(); plt.show()


In [ ]:
# QC over the tissue: are bad spots spatially clustered (e.g. tissue edges)?
import squidpy as sq
sq.pl.spatial_scatter(
    adata, color=['total_counts', 'n_genes_by_counts', 'pct_counts_mt'],
    ncols=3, size=1.3,
)


**Expected output:** three histograms (right-skewed counts/genes; mostly-low mito), then
three tissue maps. Low-count spots often sit at tissue edges/folds.

### Step 3 - Filtering
Remove spots with too few counts (poor capture) and genes seen in too few spots (cannot be
modeled reliably). Thresholds are dataset-dependent; we use gentle defaults and **explain**
rather than over-filter.

In [ ]:
n0 = adata.n_obs
sc.pp.filter_cells(adata, min_counts=500)    # drop very low-capture spots
sc.pp.filter_genes(adata, min_cells=3)        # keep genes seen in >=3 spots
# Optional: drop spots with extreme mito fraction (stressed tissue).
adata = adata[adata.obs['pct_counts_mt'] < 30].copy()
print(f'Spots: {n0:,} -> {adata.n_obs:,}; genes now: {adata.n_vars:,}')


### Step 4 - Save raw counts, then normalize + log1p
We stash the raw counts (you often need them later), then **normalize each spot to the
same total** (removes depth differences) and **log1p**-transform (compresses the dynamic
range so a few highly expressed genes do not dominate - like log-windowing intensities).

In [ ]:
adata.layers['counts'] = adata.X.copy()  # keep raw UMI counts

sc.pp.normalize_total(adata, target_sum=1e4)  # library-size normalization
sc.pp.log1p(adata)                            # log(1 + x)
adata.raw = adata  # freeze the full log-normalized matrix for plotting/markers
print('After normalize_total + log1p; .X is now log-normalized.')


### Step 5 - Highly variable genes (HVGs)
Most genes are uninformative housekeeping noise for structure discovery. HVG selection
keeps the genes that vary most across spots - **feature selection by variance**, exactly
like picking the most informative channels before clustering.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
print('HVGs selected:', int(adata.var['highly_variable'].sum()))
sc.pl.highly_variable_genes(adata)


### Step 6 - Scale + PCA
Scale each gene to unit variance (z-score; prevents high-mean genes from dominating), then
reduce to principal components. PCA is the same tool you would use to compress correlated
image features before downstream modeling.

In [ ]:
adata_hvg = adata[:, adata.var['highly_variable']].copy()
sc.pp.scale(adata_hvg, max_value=10)              # clip extreme z-scores
sc.pp.pca(adata_hvg, n_comps=50, random_state=st.SEED)

# Carry the PCA back onto the full object for later notebooks.
adata.obsm['X_pca'] = adata_hvg.obsm['X_pca']
adata.uns['pca'] = adata_hvg.uns['pca']

sc.pl.pca_variance_ratio(adata_hvg, n_pcs=50)


**Expected output:** a variance-ratio 'elbow' - the first ~20-30 PCs carry most signal,
guiding how many PCs to feed the neighbor graph in notebook 07.

### Step 7 - Save outputs (processed AnnData + QC CSV)

In [ ]:
qc_cols = ['total_counts', 'n_genes_by_counts', 'pct_counts_mt']
qc_csv = st.outputs_dir() / 'qc_summary.csv'
adata.obs[qc_cols].to_csv(qc_csv)
print('Wrote', qc_csv)

saved = st.save_adata(adata, 'adata_qc.h5ad')
print('Wrote', saved)


## Common pitfalls
- **Normalizing before QC/filtering** - compute QC on raw counts first.
- **Over-filtering** - aggressive thresholds delete real biology (e.g. low-RNA regions).
- Forgetting to keep raw counts (`layers['counts']`) - some methods need them.
- Running PCA on unscaled data - a few high-variance genes hijack the components.

## Interpretation
The matrix is now comparable across spots, log-scaled, reduced to its informative axes,
and saved. This is the substrate for clustering and spatial analysis.

## What this means biologically
Normalization removes the technical 'how much was captured here' so that differences we
see downstream reflect real differences in **which genes the tissue is using**, not depth.
Mitochondrial % flags spots where the tissue was likely damaged.

---
**Next:** `05_histology_image_loading_and_preprocessing.ipynb`.
